# Nurse Overtime and Burnout Risk Analysis
## Business Questions

This notebook uses the Gold staffing table to answer two questions:

1. Which hospitals and departments have the largest nurse staffing gaps?
2. Does understaffing increase overtime and agency-nurse use?

In [ ]:
%sql
USE CATALOG <yourcatalog>;
USE SCHEMA <yourschema>;

## Question 1: Largest nurse staffing gaps

This ranks each hospital and department by its average monthly nurse staffing gap. A larger gap means more budgeted nurse FTEs were unfilled.

In [ ]:
%sql
SELECT
  hospital_id,
  department_id,
  ROUND(AVG(nurse_staffing_gap), 2) AS avg_monthly_staffing_gap,
  ROUND(AVG(vacancy_rate) * 100, 1) AS avg_vacancy_rate_pct,
  ROUND(AVG(overtime_hours_per_filled_fte), 1) AS avg_overtime_hours_per_fte,
  ROUND(AVG(agency_hours_per_filled_fte), 1) AS avg_agency_hours_per_fte
FROM workspace.healthcare_staffing.gold_staffing_tableau_master
GROUP BY hospital_id, department_id
ORDER BY avg_monthly_staffing_gap DESC;

## Question 2: Understaffing, overtime, and agency use

Compare the results for Understaffed, At Risk, and Adequately Staffed records. Higher overtime or agency hours in the Understaffed group supports the staffing-pressure finding.

In [ ]:
%sql
SELECT
  staffing_status,
  COUNT(*) AS monthly_records,
  ROUND(AVG(nurse_staffing_gap), 2) AS avg_staffing_gap,
  ROUND(AVG(nurse_overtime_hours), 1) AS avg_overtime_hours,
  ROUND(AVG(agency_nurse_hours), 1) AS avg_agency_hours,
  ROUND(AVG(overtime_hours_per_filled_fte), 1) AS avg_overtime_hours_per_fte,
  ROUND(AVG(agency_hours_per_filled_fte), 1) AS avg_agency_hours_per_fte
FROM workspace.healthcare_staffing.gold_staffing_tableau_master
GROUP BY staffing_status
ORDER BY avg_staffing_gap DESC;

## Hospital summary for Tableau

Use this result for hospital-level KPI cards and comparison charts in Tableau.

In [ ]:
%sql
SELECT
  hospital_id,
  ROUND(AVG(nurse_staffing_gap), 2) AS avg_staffing_gap,
  ROUND(AVG(vacancy_rate) * 100, 1) AS avg_vacancy_rate_pct,
  ROUND(SUM(nurse_overtime_hours), 0) AS total_overtime_hours,
  ROUND(SUM(agency_nurse_hours), 0) AS total_agency_hours,
  ROUND(SUM(total_labor_cost), 0) AS total_labor_cost
FROM workspace.healthcare_staffing.gold_staffing_tableau_master
GROUP BY hospital_id
ORDER BY avg_staffing_gap DESC;